# Sesi 7 – Pengantar Machine Learning: Regresi Linier

| Keterangan | Detail |
|---|---|
| **Nama Lengkap** | [Farraz Dirgham H] |
| **NIM** | [240401020170] |
| **Kelas** | [IF405] |
| **Mata Kuliah** | Data Science |
| **Topik** | Pengantar Machine Learning: Regresi Linier (Simple & Multiple) |

## Pendahuluan

Sesi 7 adalah pintu masuk ke **Machine Learning (ML)**. Setelah data disiapkan dengan benar (Sesi 6), kita membangun model pertama: **Regresi Linier**.

| Tipe ML | Deskripsi | Contoh |
|---|---|---|
| **Supervised** | Data berlabel (ada target Y) | Regresi, Klasifikasi |
| **Unsupervised** | Data tanpa label | Clustering, PCA |
| **Reinforcement** | Belajar dari reward | Q-Learning |

Regresi Linier = **Supervised Learning** untuk target **numerik kontinu**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Library siap!')

## 1. Konsep Dasar Regresi Linier

**Simple:** Y = β₀ + β₁X + ε

**Multiple:** Y = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ + ε

In [ ]:
x = np.array([1,2,3,4,5,6,7,8,9,10], dtype=float)
y = 2.5*x + 5 + np.random.normal(0, 2, 10)

m = LinearRegression().fit(x.reshape(-1,1), y)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, color='steelblue', s=80, zorder=5, label='Data aktual')
xr = np.linspace(0, 11, 100)
ax.plot(xr, m.predict(xr.reshape(-1,1)), 'r-', linewidth=2.5,
        label=f'Y = {m.intercept_:.2f} + {m.coef_[0]:.2f}X')
for xi, yi in zip(x, y):
    yhat = m.predict([[xi]])[0]
    ax.plot([xi,xi],[yi,yhat],'gray',linestyle='--',linewidth=1,alpha=0.6)
ax.set_title('Konsep Regresi Linier: Garis Terbaik & Residual', fontsize=13, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('konsep_regresi.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'β₀ (Intercept): {m.intercept_:.4f}')
print(f'β₁ (Slope)    : {m.coef_[0]:.4f}')

## 2. Dataset: Prediksi Gaji Karyawan

In [ ]:
n = 350
pengalaman  = np.random.randint(0, 30, n)
pendidikan  = np.random.choice([0,1,2,3,4], n, p=[0.10,0.15,0.50,0.20,0.05])
performa    = np.random.randint(60, 101, n)
usia        = pengalaman + np.random.randint(22, 26, n)
jml_proyek  = np.random.randint(1, 20, n)

gaji = (pengalaman*0.4 + pendidikan*1.5 + performa*0.05
        + jml_proyek*0.15 + 3.5 + np.random.normal(0,0.8,n))
gaji = np.clip(gaji, 3, 30)

df = pd.DataFrame({
    'pengalaman_th' : pengalaman,
    'pendidikan_ord': pendidikan,
    'performa'      : performa,
    'usia'          : usia,
    'jml_proyek'    : jml_proyek,
    'gaji_jt'       : gaji.round(2)
})
print(f'Shape: {df.shape}')
df.describe().round(2)

## 3. EDA Sebelum Modeling

In [ ]:
fitur_list = ['pengalaman_th','pendidikan_ord','performa','usia','jml_proyek']
warna = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col, c in zip(axes.flatten()[:5], fitur_list, warna):
    ax.scatter(df[col], df['gaji_jt'], color=c, alpha=0.4, s=30)
    z  = np.polyfit(df[col], df['gaji_jt'], 1)
    xr = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(xr, np.poly1d(z)(xr), 'r-', linewidth=2)
    r = np.corrcoef(df[col], df['gaji_jt'])[0,1]
    ax.set_title(f'{col}  (r={r:.3f})', fontweight='bold', fontsize=10)
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('Gaji (Jt Rp)', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

sns.heatmap(df.corr(), annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, square=True, ax=axes[1,2], cbar_kws={'shrink':0.8})
axes[1,2].set_title('Matriks Korelasi', fontweight='bold')

plt.suptitle('EDA: Hubungan Fitur terhadap Gaji', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_gaji.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Simple Linear Regression (1 Fitur)

In [ ]:
X_s = df[['pengalaman_th']]
y   = df['gaji_jt']
X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, random_state=42)

slr = LinearRegression().fit(X_tr, y_tr)
y_pred_s = slr.predict(X_te)

print('=== Simple Linear Regression ===')
print(f'β₀ (Intercept) : {slr.intercept_:.4f}')
print(f'β₁ (Slope)     : {slr.coef_[0]:.4f}')
print(f'Persamaan      : Gaji = {slr.intercept_:.2f} + {slr.coef_[0]:.4f} × Pengalaman')
print(f'MAE  : {mean_absolute_error(y_te, y_pred_s):.4f} juta')
print(f'RMSE : {np.sqrt(mean_squared_error(y_te, y_pred_s)):.4f} juta')
print(f'R²   : {r2_score(y_te, y_pred_s):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_te, y_te, color='steelblue', alpha=0.5, s=40, label='Data Aktual')
xr = np.linspace(X_te.values.min(), X_te.values.max(), 100).reshape(-1,1)
axes[0].plot(xr, slr.predict(xr), 'r-', linewidth=2.5, label='Garis Regresi')
axes[0].set_title(f'Simple LR  |  R²={r2_score(y_te,y_pred_s):.4f}', fontweight='bold')
axes[0].set_xlabel('Pengalaman (tahun)')
axes[0].set_ylabel('Gaji (Juta Rp)')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

axes[1].scatter(y_te, y_pred_s, alpha=0.5, color='coral', s=40)
lim = [min(y_te.min(), y_pred_s.min()), max(y_te.max(), y_pred_s.max())]
axes[1].plot(lim, lim, 'b--', linewidth=2, label='Perfect Fit')
axes[1].set_title('Aktual vs Prediksi (Simple LR)', fontweight='bold')
axes[1].set_xlabel('Aktual')
axes[1].set_ylabel('Prediksi')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('simple_lr.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Multiple Linear Regression (5 Fitur)

In [ ]:
X_m = df[fitur_list]
y   = df['gaji_jt']
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_m, y, test_size=0.2, random_state=42)

pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
pipe.fit(X_tr_m, y_tr_m)
y_pred_m = pipe.predict(X_te_m)

print('=== Multiple Linear Regression ===')
coefs = pipe.named_steps['model'].coef_
for feat, coef in zip(fitur_list, coefs):
    print(f'  {feat:22s}: {coef:+.4f}')
print(f'  Intercept            : {pipe.named_steps["model"].intercept_:.4f}')
print(f'\nMAE  : {mean_absolute_error(y_te_m, y_pred_m):.4f} juta')
print(f'RMSE : {np.sqrt(mean_squared_error(y_te_m, y_pred_m)):.4f} juta')
print(f'R²   : {r2_score(y_te_m, y_pred_m):.4f}')

In [ ]:
coef_df = pd.DataFrame({'Fitur': fitur_list, 'Koefisien': coefs})
coef_df = coef_df.sort_values('Koefisien', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(coef_df['Fitur'], coef_df['Koefisien'],
        color=['#4CAF50' if c>0 else '#F44336' for c in coef_df['Koefisien']],
        edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Koefisien Multiple LR (Standardized)', fontsize=12, fontweight='bold')
ax.set_xlabel('Nilai Koefisien')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('koefisien_mlr.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Analisis Residual

In [ ]:
residuals = y_te_m.values - y_pred_m

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_pred_m, residuals, alpha=0.5, color='steelblue', s=40)
axes[0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residual vs Fitted', fontweight='bold')
axes[0].set_xlabel('Nilai Prediksi')
axes[0].set_ylabel('Residual')
axes[0].spines[['top','right']].set_visible(False)

axes[1].hist(residuals, bins=25, color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Distribusi Residual', fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].spines[['top','right']].set_visible(False)

stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot Residual', fontweight='bold')

plt.tight_layout()
plt.savefig('analisis_residual.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean Residual: {np.mean(residuals):.6f}  |  Std: {np.std(residuals):.4f}')

## 7. Cross-Validation

In [ ]:
cv_r2   = cross_val_score(pipe, X_m, y, cv=5, scoring='r2')
cv_rmse = cross_val_score(pipe, X_m, y, cv=5, scoring='neg_root_mean_squared_error')

print('=== 5-Fold Cross-Validation ===')
print(f'R² per fold    : {[round(s,4) for s in cv_r2]}')
print(f'R² Mean ± Std  : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print(f'RMSE Mean ± Std: {-cv_rmse.mean():.4f} ± {cv_rmse.std():.4f} juta')

## 8. Perbandingan Simple vs Multiple LR

In [ ]:
hasil = pd.DataFrame({
    'Model': ['Simple LR (1 fitur)','Multiple LR (5 fitur)'],
    'MAE'  : [mean_absolute_error(y_te, y_pred_s), mean_absolute_error(y_te_m, y_pred_m)],
    'RMSE' : [np.sqrt(mean_squared_error(y_te, y_pred_s)), np.sqrt(mean_squared_error(y_te_m, y_pred_m))],
    'R²'   : [r2_score(y_te, y_pred_s), r2_score(y_te_m, y_pred_m)]
})
print(hasil.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
x_pos = np.arange(2)
for i, (m, c) in enumerate(zip(['MAE','RMSE','R²'],['#2196F3','#4CAF50','#FF5722'])):
    ax.bar(x_pos + i*0.25, hasil[m], width=0.25, label=m, color=c, edgecolor='white', alpha=0.85)
ax.set_xticks(x_pos + 0.25)
ax.set_xticklabels(['Simple LR\n(1 fitur)','Multiple LR\n(5 fitur)'])
ax.set_ylabel('Nilai Metrik')
ax.set_title('Perbandingan Simple vs Multiple LR', fontweight='bold')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Prediksi Data Baru

In [ ]:
karyawan_baru = pd.DataFrame({
    'pengalaman_th' : [5, 10, 15,  3, 20],
    'pendidikan_ord': [2,  3,  3,  1,  4],
    'performa'      : [80, 90, 85, 70, 95],
    'usia'          : [27, 34, 40, 25, 45],
    'jml_proyek'    : [6, 12,  8,  3, 18]
})
karyawan_baru['prediksi_gaji_jt'] = pipe.predict(karyawan_baru).round(2)
print('=== Prediksi Gaji Karyawan Baru ===')
print(karyawan_baru.to_string(index=False))

## 10. Asumsi Regresi Linier (GAUSS-MARKOV)

| No | Asumsi | Cara Verifikasi |
|---|---|---|
| 1 | Linearitas | Scatter plot X vs Y |
| 2 | Independensi residual | Plot residual vs urutan |
| 3 | Homoskedastisitas | Residual vs Fitted |
| 4 | Normalitas Residual | Histogram / Q-Q Plot |
| 5 | Tidak ada Multikolinearitas | VIF |

## 11. Kesimpulan

| Konsep | Penjelasan |
|---|---|
| Simple LR | 1 fitur → 1 target numerik |
| Multiple LR | Banyak fitur → 1 target numerik |
| β₀ (Intercept) | Nilai Y saat semua X = 0 |
| β₁...βₙ | Perubahan Y per unit X |
| MAE | Rata-rata error absolut |
| RMSE | Error kuadrat rata-rata (sensitif outlier) |
| R² | % variansi Y yang dijelaskan model |
| Cross-Validation | Estimasi performa lebih terpercaya |

### Perjalanan Belajar Sesi 1–7
```
Sesi 1 → Pengenalan Data Science & ekosistem
Sesi 2 → Struktur data Python, NumPy, Pandas
Sesi 3 → Data Cleaning: missing, outlier, ekstraksi
Sesi 4 → Statistika dasar & analisis data
Sesi 5 → Visualisasi data
Sesi 6 → Persiapan data untuk ML
Sesi 7 → Model ML pertama: Regresi Linier end-to-end
```

> **Insight Akhir:** Regresi Linier mengajarkan fondasi ML — *fitting, generalisasi, evaluasi*, dan pentingnya memahami asumsi model. Semua model kompleks berikutnya dibangun di atas konsep ini.